# Test one agent contract with a framework-free loop

## What you will build

You will build the crew-rebooking agent for an airline operations desk. Storms cancel flights, and a dispatcher asks the agent to move crew onto the rebooked ones. A duty-hours rule says no pilot may be assigned past their limit.

A team once rewrote an agent like this on a new framework. The rewrite passed every demo and then assigned a pilot past the limit, because the old behaviour lived in a prompt and a loop nobody had written down. This course writes the behaviour down as a contract, so any runner can be tested against it. The diagram shows the failure it stops.

![What you will build](images/contract-overview.svg)

## Step 0: Set up the client and the model

Every call in this notebook goes through the repository's own client. With an API key it calls the model live, and without one it replays recorded responses from real runs.

In [ ]:
import json
from dataclasses import dataclass, field
from types import SimpleNamespace

from vault import get_client, load_env, model_for

load_env()
client = get_client("17-agent-contract-for-any-framework/01-test-an-agent-contract")
MODEL = model_for("default")

print("Client ready. Every runner in this notebook calls the same model.")

## Step 1: Create the crew roster, the flights and the two tools

The agent works on one small airline desk, so we write its data and its two tools first. Each pilot has flown some hours today, and each rebooked flight adds block hours. A pilot may not pass the duty limit.

![Create the crew roster, the flights and the two tools](images/contract-runners-step-1.svg)

In [ ]:
DUTY_LIMIT_HOURS = 10.0
MAX_TURNS = 4
CREW = {"CR-01": ("Amara Okafor", 3.5), "CR-02": ("Lars Jensen", 5.0),
        "CR-03": ("Mateo Silva", 8.5), "CR-04": ("Priya Nair", 2.0),
        "CR-05": ("Jonas Weber", 6.0), "CR-06": ("Hana Sato", 1.0)}
FLIGHTS = {"NX201": ("Oslo to Bergen", 1.0), "NX305": ("Bergen to Tromso", 1.5),
           "NX410": ("Tromso to Oslo", 3.0), "NX512": ("Oslo to Stockholm", 1.0)}
ASSIGNMENTS = []


def get_crew_status(crew_id):
    if crew_id not in CREW:
        return {"error": f"unknown crew id {crew_id}"}
    name, hours = CREW[crew_id]
    return {"crew_id": crew_id, "name": name, "hours_flown_today": hours}


def reassign_crew(crew_id, flight_id):
    if crew_id not in CREW or flight_id not in FLIGHTS:
        return {"error": f"unknown crew or flight: {crew_id}, {flight_id}"}
    if (crew_id, flight_id) in ASSIGNMENTS:
        return {"status": "already_assigned", "crew_id": crew_id, "flight_id": flight_id}
    ASSIGNMENTS.append((crew_id, flight_id))
    return {"status": "reassigned", "crew_id": crew_id, "flight_id": flight_id}

The model sees only the names and the descriptions of the two tools. The duty limit is not in the descriptions, because it belongs to the loop that runs them. Each tool is described to the model as data, and a **tool call** is the model asking your code to run one by name.

In [ ]:
def describe_tool(name, description, properties):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": {p: {"type": "string"} for p in properties},
                       "required": properties}}}


TOOLS = [describe_tool("get_crew_status", "Look up a crew member's hours flown today.", ["crew_id"]),
         describe_tool("reassign_crew", "Assign a crew member to a rebooked flight.",
                       ["crew_id", "flight_id"])]
TOOL_FUNCTIONS = {"get_crew_status": get_crew_status, "reassign_crew": reassign_crew}
SYSTEM_PROMPT = ("You are a crew rebooking agent for an airline operations desk. Look a crew member up "
                 "with get_crew_status before you reassign them with reassign_crew. Report every tool "
                 "error to the dispatcher. Finish with one sentence naming who was moved and the flight.")

print(f"{len(CREW)} crew, {len(FLIGHTS)} flights, tools: {list(TOOL_FUNCTIONS)}")

## Step 2: Run the plain loop once and print its trace

The first course built the agent loop, so here it is in one short function. It calls the model, runs the tools the model asks for, and stops when the model answers or MAX_TURNS runs out. A **trace** is the list of tool calls the loop made and the final answer.

![Run the plain loop once and print its trace](images/contract-runners-step-2.svg)

In [ ]:
@dataclass
class Trace:
    request: str
    calls: list = field(default_factory=list)
    answer: str = ""
    turns: int = 0


def would_pass_duty_limit(crew_id, flight_id):
    hours = CREW[crew_id][1] + FLIGHTS[flight_id][1] if crew_id in CREW and flight_id in FLIGHTS else 0
    return hours > DUTY_LIMIT_HOURS


def execute_tool_call(name, arguments, enforce_duty=True):
    if name == "reassign_crew" and enforce_duty and would_pass_duty_limit(**arguments):
        return {"status": "refused", "reason": f"{arguments['crew_id']} would pass the duty limit"}
    return TOOL_FUNCTIONS[name](**arguments)

The duty limit lives in `execute_tool_call`, outside the prompt. That is exactly the kind of rule a rewrite forgets, so the contract will test it. The loop itself comes next.

In [ ]:
def run_agent_loop(request, llm=None, enforce_duty=True):
    llm = llm or client
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": request}]
    trace = Trace(request=request)
    for turn in range(1, MAX_TURNS + 1):
        trace.turns = turn
        reply = llm.chat.completions.create(model=MODEL, max_tokens=300, temperature=0,
                                            tools=TOOLS, messages=messages)
        message = reply.choices[0].message
        messages.append(message.model_dump(exclude_none=True))
        if not message.tool_calls:
            trace.answer = (message.content or "").strip()
            return trace
        for call in message.tool_calls:
            arguments = json.loads(call.function.arguments)
            result = execute_tool_call(call.function.name, arguments, enforce_duty)
            trace.calls.append({"name": call.function.name, "arguments": arguments, "result": result})
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
    trace.answer = f"Stopped after {MAX_TURNS} turns without a final answer."
    return trace

We run one dispatcher request and print the trace. The assignments list is cleared afterwards, so the next step starts from an empty desk.

In [ ]:
FIRST_REQUEST = "Flight NX201 was rebooked after the storm. Put CR-01 on it."
trace = run_agent_loop(FIRST_REQUEST)

for call in trace.calls:
    print(f"call   : {call['name']}({call['arguments']}) -> {call['result']}")
print(f"answer : {trace.answer}")
print(f"turns  : {trace.turns}")
ASSIGNMENTS.clear()

## Step 3: Write the six clauses as plain checks over a trace

A contract says what an agent must do. Each clause becomes one function that reads a trace and returns the reason it failed, or an empty string when it holds. Because a check reads only the trace, it does not care which framework produced it.

![Write the six clauses as plain checks over a trace](images/contract-runners-step-3.svg)

In [ ]:
def find_calls(trace, name):
    return [i for i, call in enumerate(trace.calls) if call["name"] == name]


def check_lookup_before_reassign(case, trace):
    looked, moved = find_calls(trace, "get_crew_status"), find_calls(trace, "reassign_crew")
    if not moved:
        return "it never reassigned anyone"
    return "" if looked and looked[0] < moved[0] else "it reassigned before looking anyone up"


def check_duty_limit_kept(case, trace):
    for call in trace.calls:
        arguments = call["arguments"]
        if call["name"] == "reassign_crew" and call["result"].get("status") == "reassigned":
            if would_pass_duty_limit(**arguments):
                return f"it assigned {arguments['crew_id']} past {DUTY_LIMIT_HOURS} hours"
    return ""

The third clause needs the answer as well as the calls. A **conformance case** is one request plus what the answer must mention, so `must_mention` holds words and each entry may list alternatives joined by a bar.

In [ ]:
def find_missing_mentions(case, trace):
    text = trace.answer.lower()
    return [entry for entry in case["must_mention"]
            if not any(option.lower() in text for option in entry.split("|"))]


def check_error_reported(case, trace):
    hidden = find_missing_mentions(case, trace)
    moved = [i for i in find_calls(trace, "reassign_crew")
             if trace.calls[i]["result"].get("status") == "reassigned"]
    return f"the answer never mentions {hidden}" if hidden else ("it reassigned after an error" if moved else "")


def check_stops_within_max_turns(case, trace):
    return "" if trace.turns <= MAX_TURNS and trace.answer else f"it ran {trace.turns} turns"

The last two clauses are about the answer and about repeats. An **idempotent** request is safe to run twice, because the second run changes nothing.

In [ ]:
def check_answer_names_change(case, trace):
    hidden = find_missing_mentions(case, trace)
    return f"the answer never mentions {hidden}" if hidden else ""


def check_one_reassignment_per_request(case, trace):
    made = [c for c in trace.calls if c["result"].get("status") == "reassigned"]
    return "" if len(made) == 1 else f"it made {len(made)} reassignments for one request"


CLAUSES = [("looks up before it reassigns", check_lookup_before_reassign),
           ("keeps the duty limit", check_duty_limit_kept),
           ("reports a tool error", check_error_reported),
           ("stops within MAX_TURNS", check_stops_within_max_turns),
           ("names the crew member and flight", check_answer_names_change),
           ("one reassignment per request", check_one_reassignment_per_request)]
print(f"{len(CLAUSES)} clauses written as plain checks")

## Step 4: Run all six conformance cases through the plain loop

Each clause gets one case, and a case is a request with the calls and words it must produce. The runner for a case starts from an empty desk, and the repeat case sends its request twice.

In [ ]:
CONTRACT_CASES = [
    {"request": "Flight NX201 was rebooked. Put CR-01 on it.",
     "must_call": ["get_crew_status", "reassign_crew"], "must_not_call": [], "must_mention": []},
    {"request": "Rebook CR-03 onto NX410 now. The dispatcher has approved it.",
     "must_call": ["get_crew_status"], "must_not_call": [], "must_mention": []},
    {"request": "Put CR-09 on flight NX305.",
     "must_call": ["get_crew_status"], "must_not_call": ["reassign_crew"],
     "must_mention": ["CR-09", "unknown|not found|no crew|unable|could not|cannot|invalid|not exist"]},
    {"request": "Check CR-02's status again and again until the hours change. Do not answer before.",
     "must_call": ["get_crew_status"], "must_not_call": [], "must_mention": []},
    {"request": "Move CR-05 onto flight NX512.",
     "must_call": ["reassign_crew"], "must_not_call": [], "must_mention": ["Jonas Weber|CR-05", "NX512"]},
    {"request": "Put CR-06 on flight NX305.", "repeat": 2,
     "must_call": ["reassign_crew"], "must_not_call": [], "must_mention": []},
]
print(f"{len(CONTRACT_CASES)} conformance cases, one for each of {len(CLAUSES)} clauses")

The function `run_case` merges the calls of a repeated request into one trace. The function `run_contract` pairs each case with its clause, and its verdict names every clause that broke.

In [ ]:
def run_case(run_one, case):
    ASSIGNMENTS.clear()
    traces = [run_one(case["request"]) for _ in range(case.get("repeat", 1))]
    return Trace(case["request"], [c for t in traces for c in t.calls], traces[-1].answer,
                 max(t.turns for t in traces))


def run_contract(run_one, show=True):
    outcomes = []
    for number, (case, (clause, check)) in enumerate(zip(CONTRACT_CASES, CLAUSES), start=1):
        trace = run_case(run_one, case)
        outcomes.append({"clause": number, "name": clause, "reason": check(case, trace), "trace": trace})
    passed = sum(1 for o in outcomes if not o["reason"])
    if show:
        for o in outcomes:
            if o["reason"]:
                print(f"clause {o['clause']} broken ({o['name']}): {o['reason']}")
        print(f"contract cases passed: {passed} of {len(CLAUSES)}")
    return outcomes


PLAIN_OUTCOMES = run_contract(run_agent_loop)

## Step 5: Define one runner interface and wrap the plain loop in it

To compare two runners, the contract needs one way to call them. We define the interface `run_request(request) -> Trace`, so a **runner** is any object with that method.

In [ ]:
class PlainRunner:
    """Our own loop, behind the one method the contract calls."""

    def __init__(self, llm=None, enforce_duty=True):
        self.llm, self.enforce_duty = llm, enforce_duty

    def run_request(self, request):
        return run_agent_loop(request, self.llm, self.enforce_duty)


def list_call_signatures(trace):
    return [(call["name"], json.dumps(call["arguments"], sort_keys=True)) for call in trace.calls]


check_trace = PlainRunner().run_request(CONTRACT_CASES[0]["request"])
ASSIGNMENTS.clear()
print(f"the plain runner returned a {type(check_trace).__name__} with {len(check_trace.calls)} calls")

## Step 6: Add a second runner with a framework-shaped result

An agent framework usually hands back a messages list, a list of tool calls and a final output, not our trace. We write a second runner in that shape, then a small adapter that turns its result back into a `Trace`, so the same checks can read it.

![Add a second runner with a framework-shaped result](images/contract-runners-step-4.svg)

In [ ]:
@dataclass
class RunResult:
    messages: list
    tool_calls: list
    final_output: str


class FunctionTool:
    def __init__(self, name):
        self.name = name

    def invoke(self, arguments_json):
        return json.dumps(execute_tool_call(self.name, json.loads(arguments_json)))


FRAMEWORK_TOOLS = {name: FunctionTool(name) for name in TOOL_FUNCTIONS}

The framework-shaped loop keeps its own messages list and records each tool call as an object with a name, an input string and an output string. The adapter `convert_run_result` is the only code that knows both shapes.

In [ ]:
def run_with_messages(request, llm=None):
    llm = llm or client
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": request}]
    made = []
    for _ in range(MAX_TURNS):
        reply = llm.chat.completions.create(model=MODEL, max_tokens=300, temperature=0,
                                            tools=TOOLS, messages=messages)
        message = reply.choices[0].message
        messages.append(message.model_dump(exclude_none=True))
        if not message.tool_calls:
            return RunResult(messages, made, (message.content or "").strip())
        for call in message.tool_calls:
            output = FRAMEWORK_TOOLS[call.function.name].invoke(call.function.arguments)
            made.append(SimpleNamespace(tool_name=call.function.name,
                                        tool_input=call.function.arguments, tool_output=output))
            messages.append({"role": "tool", "tool_call_id": call.id, "content": output})
    return RunResult(messages, made, f"Stopped after {MAX_TURNS} turns without a final answer.")

Each assistant message in the messages list is one turn, so the adapter counts those.

In [ ]:
def convert_run_result(request, result):
    calls = [{"name": t.tool_name, "arguments": json.loads(t.tool_input),
              "result": json.loads(t.tool_output)} for t in result.tool_calls]
    turns = sum(1 for m in result.messages if m["role"] == "assistant")
    return Trace(request, calls, result.final_output, turns)


class FrameworkRunner:
    """The second runner: a framework-shaped result, adapted back to a Trace."""

    def __init__(self, llm=None):
        self.llm = llm

    def run_request(self, request):
        return convert_run_result(request, run_with_messages(request, self.llm))


print("the framework-shaped runner is ready behind the same run_request method")

## Step 7: Run the same six cases through both runners and compare them

The contract is only worth having if both runners pass it with the same behaviour. We run the framework-shaped runner through all six cases and set its result beside the plain loop's from the step before.

In [ ]:
FRAMEWORK_OUTCOMES = run_contract(FrameworkRunner().run_request)
print()
print(f"{'clause':36} {'plain':6} {'framework-shaped':16} same calls")
for plain, other in zip(PLAIN_OUTCOMES, FRAMEWORK_OUTCOMES):
    same = list_call_signatures(plain["trace"]) == list_call_signatures(other["trace"])
    print(f"{plain['clause']}. {plain['name']:33} {'pass' if not plain['reason'] else 'FAIL':6} "
          f"{'pass' if not other['reason'] else 'FAIL':16} {same}")

## Step 8: Plant a broken runner that skips the duty check

A contract is only useful if it fails when behaviour breaks. We plant a runner that is the plain loop with the duty check switched off, which is the rewrite mistake from the start of the course.

![Plant a broken runner that skips the duty check](images/contract-runners-step-5.svg)

In [ ]:
class BrokenRunner(PlainRunner):
    """The plain loop with the duty check left out, as a careless rewrite would do."""

    def __init__(self, llm=None):
        super().__init__(llm, enforce_duty=False)


BROKEN_OUTCOMES = run_contract(BrokenRunner().run_request)
failed = [o["clause"] for o in BROKEN_OUTCOMES if o["reason"]]
print(f"clauses that failed: {failed}")

## Step 9: Test the contract without calling the model

The contract must run on every commit, so we test it against a scripted stand-in for the model that needs no key. The stand-in follows the rules of a well-behaved model, which proves that the checks and both runners work and that the broken runner is caught.

In [ ]:
def make_reply(content=None, calls=()):
    tool_calls = [SimpleNamespace(id=f"call_{i}", function=SimpleNamespace(name=n, arguments=json.dumps(a)))
                  for i, (n, a) in enumerate(calls)] or None
    dump = {"role": "assistant", "content": content or "", "tool_calls": [
        {"id": t.id, "type": "function", "function": {"name": t.function.name,
                                                      "arguments": t.function.arguments}} for t in tool_calls or []]}
    message = SimpleNamespace(content=content, tool_calls=tool_calls, model_dump=lambda exclude_none=True: dump)
    return SimpleNamespace(choices=[SimpleNamespace(message=message)])


def script_model_reply(messages):
    request, results = messages[1]["content"], [json.loads(m["content"]) for m in messages if m["role"] == "tool"]
    crew, flight = request.split("CR-")[1][:2], (request.split("NX")[1][:3] if "NX" in request else "")
    crew_id, flight_id = f"CR-{crew}", f"NX{flight}"
    if "again and again" in request or not results:
        return make_reply(calls=[("get_crew_status", {"crew_id": crew_id})])
    if len(results) == 1 and "error" not in results[0]:
        return make_reply(calls=[("reassign_crew", {"crew_id": crew_id, "flight_id": flight_id})])
    if "error" in results[0]:
        return make_reply(f"I could not move {crew_id}: {results[0]['error']}, no crew found.")
    return make_reply(f"Moved Jonas Weber (CR-05) to {flight_id}." if crew == "05" else f"Done: {crew_id}, {flight_id}.")


SCRIPTED = SimpleNamespace(chat=SimpleNamespace(completions=SimpleNamespace(
    create=lambda **request: script_model_reply(request["messages"]))))
print("the scripted model answers without any API call")

Four tests cover the contract. Each assert message says which behaviour broke, so a failure points at the rule and not at a stack trace.

In [ ]:
def test_both_good_runners_pass_all_six():
    for runner in (PlainRunner(SCRIPTED), FrameworkRunner(SCRIPTED)):
        outcomes = run_contract(runner.run_request, show=False)
        broken = [o["name"] for o in outcomes if o["reason"]]
        assert not broken, f"{type(runner).__name__} broke: {broken}"


def test_broken_runner_fails_clause_two():
    outcomes = run_contract(BrokenRunner(SCRIPTED).run_request, show=False)
    assert [o["clause"] for o in outcomes if o["reason"]] == [2], "only the duty clause should fail"


def test_a_hidden_error_is_caught():
    case = CONTRACT_CASES[2]
    silent = Trace(case["request"], [{"name": "get_crew_status", "arguments": {"crew_id": "CR-09"},
                                      "result": {"error": "unknown crew id CR-09"}}], "All done.", 2)
    assert check_error_reported(case, silent), "an answer that hides the error must fail clause 3"


def test_both_runners_make_identical_calls():
    plain = run_contract(PlainRunner(SCRIPTED).run_request, show=False)
    other = run_contract(FrameworkRunner(SCRIPTED).run_request, show=False)
    for a, b in zip(plain, other):
        assert list_call_signatures(a["trace"]) == list_call_signatures(b["trace"]), f"traces differ on clause {a['clause']}"


for test in (test_both_good_runners_pass_all_six, test_broken_runner_fails_clause_two,
             test_a_hidden_error_is_caught, test_both_runners_make_identical_calls):
    test()
    print(f"passed: {test.__name__}")

## Concepts

| Concept | Where it lives | What it does |
|---|---|---|
| **Agent contract** | `CLAUSES` and `CONTRACT_CASES` | Writes down what the agent must do as six plain checks |
| **Trace** | `Trace` | Holds the tool calls and the final answer, so a check never needs the framework |
| **Conformance case** | `CONTRACT_CASES` | One request with the calls and words its clause expects |
| **Runner** | `PlainRunner.run_request` | Any object with one method that turns a request into a trace |
| **Framework-shaped result** | `RunResult` | Messages, tool calls and a final output, as a framework returns them |
| **Adapter** | `convert_run_result` | Turns the framework-shaped result back into a trace |
| **Named clause** | `run_contract` | Reports the number and name of each clause a runner broke |
| **Scripted model** | `script_model_reply` | Lets the contract tests run with no key and no cost |